# Notebook B2: controls for steering, and *where* the count is read (GPU, about 1-1.2 h)

Notebook B found: (1) the history effect reaches the last-position state over layers 14-20; (2) adding the count ("prior") direction at layers 16-24 moves $\Delta$ strongly, but the shuffled-label direction does the same with opposite sign and the evidence direction too. All three act through one shared axis (the answer read-out), so that experiment does not show the count direction is special.

**Part 1: steering with proper controls** (layers 16, 20, 24; size 1x; both signs; same 40 images as B)
- `shift`: the natural direction (reference; moved $\Delta$ by 76% of the natural jump at layer 20).
- `shiftshuf1..6`: the same recipe with the count labels shuffled among trials (6 different shuffles = a null distribution for the shift effect).
- `*_perp_evid`, `*_perp_read`: the shift / count directions with the evidence axis, or the answer read-out axis (from the unembedding), projected out. If the effect survives, it is not just the read-out axis.
- cosines of every direction with the read-out axis and the evidence axis are printed.

**Part 2: where is the count read? (token-position patching)** Two history prompts for the same image, "left 8 / right 2" vs "left 2 / right 8", have the same token length and differ only at the two digit tokens. We run one prompt and overwrite the residual stream at chosen positions with the other prompt's, at layer $L$:
- `digits`: the two count tokens; `words`: the rest of the history sentence; `last`: the answer position (as in B); `post_hist`: everything after the image (positive control: must give recovery ~1 at early layers).
- Recovery = how much of the gap $\Delta(8)-\Delta(2)$ moves. Digits should matter at early layers and stop mattering once the count has been copied elsewhere; the last position takes over later. The crossover tells us the layer where the count is read.

**Setup:** GPU on (2x T4), Internet on, dataset `llava-pve-outputs-run` attached, Kaggle secret `KAGGLE_KEY` on. Use *Save & Run All (Commit)*. Checkpoints go to `ponishbhatia3/llava-pve-b2-ckpt` and resume if the session dies.
**Tested on a mock model, not on the real LLaVA. Self-test cells stop with a clear message if a hook or a position is wrong.**

In [1]:
# ===================== CONFIG =====================
MOCK = False
QUICK = False                # True = tiny smoke test (~6 min)
SEED = 0                     # must match the main run
DATA = "/kaggle/input/datasets/ponishbhatia3/llava-pve-outputs-run"
OUT = "/kaggle/working/B2_out"
N_STEER = 40                                 # images for both parts (same 40 as notebook B)
STEER2_LAYERS = [16, 20, 24]
NSHUF = 6                                    # shuffled-label shift directions (null distribution)
P_LAYERS = [4, 8, 10, 12, 14, 16, 18, 20, 24]
P_SETS = ["digits", "words", "last", "post_hist"]
DIRS = ["shift", "prior", "evid", "shuf", "rand1"]          # needed by the shared direction code
if QUICK:
    N_STEER, STEER2_LAYERS, NSHUF, P_LAYERS = 12, [20], 2, [8, 14, 20]
import os, re, glob, json, math, time, zipfile, shutil, warnings
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from PIL import Image
warnings.filterwarnings("ignore")
os.makedirs(OUT, exist_ok=True)
SUMMARY = []
def say(s=""): print(s); SUMMARY.append(s)

In [2]:
# ===================== CHECKPOINTS TO A KAGGLE DATASET (so a dead session loses nothing) =====================
KAGGLE_USER = "ponishbhatia3"
CKPT_SLUG = f"{KAGGLE_USER}/llava-pve-b2-ckpt"          # private dataset, created on the first push, new version afterwards
PUSH = not MOCK                                         # set False to disable (then only /kaggle/working is saved)
PUSH_EVERY_MIN = 10
CKPT_DIR = "/kaggle/working/B2_ckpt"
RUN_CFG = dict(SEED=SEED, QUICK=QUICK, N_STEER=N_STEER, STEER2_LAYERS=STEER2_LAYERS, NSHUF=NSHUF, P_LAYERS=P_LAYERS, P_SETS=P_SETS)
json.dump(RUN_CFG, open(f"{OUT}/run_config.json", "w"))
_last_push = 0.0

def _setup_creds():
    assert KAGGLE_USER and "YOUR" not in KAGGLE_USER.upper(), "KAGGLE_USER is a placeholder"
    from kaggle_secrets import UserSecretsClient
    sc, key = UserSecretsClient(), None
    for name in ("KAGGLE_KEY", "KAGGLE_API_TOKEN"):
        try: key = sc.get_secret(name); break
        except Exception: pass
    assert key, "No Kaggle secret found. Add-ons -> Secrets -> add KAGGLE_KEY (your Kaggle API key / token from Settings -> API), switch it ON for this notebook, rerun. Or set PUSH = False."
    os.environ["KAGGLE_USERNAME"] = KAGGLE_USER
    os.environ["KAGGLE_API_TOKEN" if key.startswith("KGAT_") else "KAGGLE_KEY"] = key

def _api():
    from kaggle.api.kaggle_api_extended import KaggleApi
    a = KaggleApi(); a.authenticate(); return a

def push_ckpt(note):
    os.makedirs(CKPT_DIR, exist_ok=True)
    open(f"{CKPT_DIR}/README.txt", "w").write("Checkpoints of notebook B2 (steering controls, token-position patching). Resume by attaching this dataset as input and rerunning with the same settings.\n")
    for pat in ("*.csv", "*.png", "*.txt", "*.json"):
        for f in glob.glob(f"{OUT}/{pat}"): shutil.copy(f, CKPT_DIR)
    json.dump({"title": "LLaVA PVE B2 checkpoints", "id": CKPT_SLUG, "licenses": [{"name": "CC0-1.0"}]}, open(f"{CKPT_DIR}/dataset-metadata.json", "w"))
    api = _api()
    try: api.dataset_status(CKPT_SLUG); exists = True
    except Exception: exists = False
    res = api.dataset_create_version(CKPT_DIR, note, quiet=True, dir_mode="zip") if exists else api.dataset_create_new(CKPT_DIR, public=False, quiet=True, dir_mode="zip")
    err = getattr(res, "error", None)
    assert not err, f"Kaggle returned an error: {err}"
    print(f"[checkpoint] {'new version' if exists else 'created dataset'} {CKPT_SLUG} ({note}) | server says: {getattr(res, 'status', res)}", flush=True)

def maybe_push(note, force=False):
    global _last_push
    if not PUSH or (not force and time.time() - _last_push < PUSH_EVERY_MIN * 60): return
    try: push_ckpt(note); _last_push = time.time()
    except Exception as e: print(f"!!! CHECKPOINT PUSH FAILED (the run continues; files are still in {OUT}): {type(e).__name__}: {str(e)[:300]}", flush=True)

# resume: if a previous checkpoint dataset is attached as input with the SAME settings, copy its files back
for hit in glob.glob("/kaggle/input/**/run_config.json", recursive=True):
    try:
        if "llava-pve-outputs-run" in hit or json.load(open(hit)) != json.loads(json.dumps(RUN_CFG)): continue
    except Exception: continue
    d = os.path.dirname(hit)
    for name in ("steer2_rows.csv", "tokpatch_rows.csv"):
        if os.path.exists(f"{d}/{name}") and not os.path.exists(f"{OUT}/{name}"): shutil.copy(f"{d}/{name}", f"{OUT}/{name}"); print("resumed", name, "from", d)

if PUSH:                                                # fail fast: a real test push now, not after 40 minutes
    _setup_creds(); push_ckpt("preflight")
    print("checkpoint push works; later pushes happen every", PUSH_EVERY_MIN, "min and at the end of each stage")

[checkpoint] new version ponishbhatia3/llava-pve-b2-ckpt (preflight) | server says: Ok
checkpoint push works; later pushes happen every 10 min and at the end of each stage


In [3]:
# ===================== trials, prompts, images: identical to the main run (counts10 history) =====================
CONTRASTS = [0.0, 0.03, 0.045, 0.0625, 0.09, 0.125, 1.0]; N_FULL, FULL_UNBIASED, WINDOW = 600, 90, 10
SIZE = 336; _Y, _X = np.mgrid[0:SIZE, 0:SIZE].astype(np.float32)
def make_trials(n, seed, contrasts, n_unbiased):
    rng = np.random.default_rng(seed); p = np.full(n, 0.5)
    t, cur = n_unbiased, float(rng.choice([0.2, 0.8]))
    while t < n:
        L = 20 + int(rng.exponential(60))
        while L > 100: L = 20 + int(rng.exponential(60))
        p[t:t + L] = cur; t, cur = t + L, round(1.0 - cur, 2)
    side = np.where(rng.random(n) < p, "right", "left"); c = rng.choice(contrasts, n)
    return dict(n=n, p_right=p, side=side, contrast=c, dx=rng.uniform(-15, 15, n), dy=rng.uniform(-30, 30, n), phi=rng.uniform(0, 2 * np.pi, n))
def render(T, t, seed):
    cx = SIZE / 2 + (84 if T["side"][t] == "right" else -84) + T["dx"][t]; cy = SIZE / 2 + T["dy"][t]
    env = np.exp(-((_X - cx) ** 2 + (_Y - cy) ** 2) / (2 * 36 ** 2))
    a = 0.5 + 0.5 * T["contrast"][t] * env * np.sin(2 * np.pi * (_X - cx) / 24 + T["phi"][t])
    a = a + np.random.default_rng(seed * 100003 + t).normal(0, 0.02, a.shape)
    return Image.fromarray(np.stack([(np.clip(a, 0, 1) * 255).astype(np.uint8)] * 3, -1))
def hist_counts(T, t, w):
    past = T["side"][max(0, t - w):t]; return len(past), int((past == "right").sum())
def history_text(T, t):
    n, nr = hist_counts(T, t, WINDOW)
    return "No previous trials." if n == 0 else f"In the last {n} trials the grating was on the left {n - nr} times and on the right {nr} times."
QS = ["Is the grating on the left or right of the image? Answer with one word.", "Is the grating on the right or left of the image? Answer with one word."]
def prompt(hist, q): return f"USER: <image>\n{hist}\n{q} ASSISTANT:" if hist else f"USER: <image>\n{q} ASSISTANT:"
T = make_trials(N_FULL, SEED + 1, CONTRASTS, FULL_UNBIASED)

In [4]:
# ===================== cache + directions (recomputed from the cached residual streams, same recipe as notebook A) =====================
ROWS = pd.read_csv(glob.glob(f"{DATA}/**/full_rows_all.csv", recursive=True)[0])
HS = np.concatenate([np.load(f, mmap_mode="r") for f in sorted(glob.glob(f"{DATA}/**/full_hs_*.npy", recursive=True))])    # (trial, [none,hist], layer 0..32, dim)
NTR, _, NLAY, D = HS.shape
w_ = ROWS.pivot(index="t", columns="order", values="delta")
M = ROWS[ROWS.order == "none"].set_index("t").loc[w_.index][["side", "contrast", "p_right", "n_hist", "n_hist_right"]].copy()
M["d_none"], M["d_hist"] = w_["none"], w_["hist"]; M["shift"] = M.d_hist - M.d_none
assert (M.side.values == T["side"]).all() and np.allclose(M.contrast.values, T["contrast"]), "regenerated trials differ from the main run: check SEED / CONTRASTS"
NR = M.n_hist_right.values.astype(int); FULL = (M.n_hist.values == 10)
frac = (NR + .5) / (M.n_hist.values + 1); TP = np.log(frac / (1 - frac))
cell = pd.get_dummies(M.side.astype(str) + "|" + M.contrast.astype(str), dtype=float).values
lsq = lambda A, B: np.linalg.lstsq(A, B, rcond=None)[0]
def count_dir(X, target, Cc):
    Xr = X - Cc @ lsq(Cc, X); yt = target - Cc @ lsq(Cc, target); d = Xr.T @ yt; return d / (np.linalg.norm(d) + 1e-12)
hi_c = M.contrast.values >= 0.09; sr = M.side.values == "right"
hi_cnt, lo_cnt = FULL & (NR >= 8), FULL & (NR <= 2)
rng0 = np.random.default_rng(0); perm = rng0.permutation(FULL.sum())
DIRVEC, NORM, RESNORM = {}, np.zeros(NLAY), np.zeros(NLAY)
for name in DIRS: DIRVEC[name] = np.zeros((NLAY, D), np.float32)
UNIT = {name: np.zeros((NLAY, D), np.float32) for name in ["shift", "prior", "evid", "shuf", "rand1", "rand2"]}    # unit-norm versions, always computed
for L in range(1, NLAY):
    Xn, Xh = HS[:, 0, L, :].astype(np.float32), HS[:, 1, L, :].astype(np.float32)
    shift = ((Xh - Xn)[hi_cnt].mean(0) - (Xh - Xn)[lo_cnt].mean(0)) / 2          # natural half-jump from a no-history state to a high/low-count state
    NORM[L] = np.linalg.norm(shift); RESNORM[L] = np.linalg.norm(Xn, axis=1).mean(); unit = lambda v: v / (np.linalg.norm(v) + 1e-12)
    cand = {"shift": unit(shift),
            "prior": count_dir(Xh[FULL], TP[FULL], cell[FULL]),
            "evid": unit(Xn[hi_c & sr].mean(0) - Xn[hi_c & ~sr].mean(0)),
            "shuf": count_dir(Xh[FULL], TP[FULL][perm], cell[FULL]),
            "rand1": unit(np.random.default_rng(11 + L).normal(size=D)), "rand2": unit(np.random.default_rng(97 + L).normal(size=D))}
    for name in cand: UNIT[name][L] = cand[name]
    for name in DIRS: DIRVEC[name][L] = cand[name] * NORM[L]                     # every direction gets the SAME norm
nat_hi = M["shift"][hi_cnt].mean(); nat_lo = M["shift"][lo_cnt].mean(); NAT_HALF = (nat_hi - nat_lo) / 2
print(f"cache {HS.shape} | natural behavioral half-jump in delta (high-count minus low-count)/2 = {NAT_HALF:.2f} logits")
print("natural half-jump norm | typical residual norm, per layer:", {L: (round(float(NORM[L]), 2), round(float(RESNORM[L]), 1)) for L in (8, 10, 12, 14, 16, 20, 24)})
print("cos(count direction, evidence direction) at layers 8/12/16/20:", {L: round(float(UNIT['prior'][L] @ UNIT['evid'][L]), 2) for L in (8, 12, 16, 20)})

cache (600, 2, 33, 4096) | natural behavioral half-jump in delta (high-count minus low-count)/2 = 0.33 logits
natural half-jump norm | typical residual norm, per layer: {8: (0.05, 9.3), 10: (0.11, 12.8), 12: (0.61, 17.1), 14: (1.09, 20.7), 16: (1.87, 27.3), 20: (3.18, 54.2), 24: (3.66, 87.6)}
cos(count direction, evidence direction) at layers 8/12/16/20: {8: -0.01, 12: 0.06, 16: 0.42, 20: 0.69}


In [5]:
# ===================== model runner =====================
if MOCK:
    class Runner:
        """Numpy stand-in: state at layer L is linear in (evidence, count); delta reads it out. Pipeline test only."""
        def __init__(s): s.rng = np.random.default_rng(5)
        def prep(s, img, text):
            a = np.abs(np.asarray(img)[..., 0].astype(float) - 128); cx = (a.sum(0) * np.arange(SIZE)).sum() / (a.sum() + 1e-6)
            m = re.search(r"on the right (\d+) times", text); n = None
            if m: nr = int(m.group(1)); nl = int(re.search(r"on the left (\d+) times", text).group(1)); n = (nr + .5) / (nr + nl + 1)
            return dict(ev=float(np.clip((cx - 168) * 0.03, -1.7, 1.7)), frac=n)
        def state(s, inp, L):
            x = np.zeros(D, np.float32); h = np.random.default_rng(int(abs(inp["ev"]) * 1e4) % 99991)
            x += 0.02 * h.normal(size=D)
            if L >= 4: x[0] += inp["ev"] * (0.5 if inp["frac"] is not None else 1.0)
            if L >= 8 and inp["frac"] is not None: x[1] += (inp["frac"] - .5) * 3; x[0] += (inp["frac"] - .5) * 0.5
            return x
        def readout(s, x): return -0.7 + 3.0 * x[0] + 1.0 * x[1]
        def delta(s, inp, edit=None):
            x = s.state(inp, 32)
            if edit is not None:
                L, mode, vec = edit; xl = s.state(inp, L)
                x = x + ((vec - xl) if mode == "replace" else vec)
            return float(s.readout(x))
        def hs(s, inp): return np.stack([s.state(inp, L) for L in range(NLAY)]).astype(np.float16)
        def readout_axis(s): a = np.zeros(D, np.float32); a[0], a[1] = 3.0, 1.0; return a
        def positions(s, ia, ib, inone): return dict(digits=["d"], words=["w"], last=["l"], post_hist=["d", "w", "l"])
        def capture(s, inp, Ls, pos): return {"inp": inp}
        def delta_pos(s, inp, L, pos, cap, rowmap):               # mock: each position set carries a layer-dependent fraction of the donor effect
            f = {"d": 1.0 if L <= 10 else 0.1, "w": 0.2 if L <= 10 else 0.0, "l": 0.0 if L < 14 else min(1.0, (L - 14) / 6)}
            fr = 1.0 if len(pos) > 1 else f[pos[0]]; a, b = s.delta(inp), s.delta(cap["inp"]); return a + fr * (b - a)
    RUN = Runner()
else:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes", "accelerate"], check=False)
    import torch
    from transformers import AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig
    NAME = "llava-hf/llava-1.5-7b-hf"
    model = LlavaForConditionalGeneration.from_pretrained(NAME, torch_dtype=torch.float16, device_map="auto",
                quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)).eval()
    proc = AutoProcessor.from_pretrained(NAME)
    L_ID, R_ID = [proc.tokenizer.encode(w, add_special_tokens=False)[0] for w in ("Left", "Right")]
    _lists = [(n, m) for n, m in model.named_modules() if isinstance(m, torch.nn.ModuleList) and len(m) == 32]
    assert len(_lists) == 1, f"expected exactly one 32-layer list, found {[n for n, _ in _lists]}"
    LAYERS = _lists[0][1]; print("language-model layers found at:", _lists[0][0])
    class Runner:
        def prep(s, img, text): return proc(images=img, text=text, return_tensors="pt").to(model.device, torch.float16)
        def delta(s, inp, edit=None):
            handle = None
            if edit is not None:
                L, mode, vec = edit; v0 = torch.tensor(vec, dtype=torch.float16)
                def hook(mod, args, output):                      # output of decoder layer L-1 == hidden_states[L]
                    h = output[0] if isinstance(output, tuple) else output
                    v = v0.to(h.device, h.dtype)                  # the model can be split over 2 GPUs: put the vector where this layer lives
                    h = h.clone()
                    if mode == "replace": h[:, -1, :] = v
                    else: h[:, -1, :] = h[:, -1, :] + v
                    return (h,) + tuple(output[1:]) if isinstance(output, tuple) else h
                handle = LAYERS[L - 1].register_forward_hook(hook)
            try:
                with torch.no_grad(): lg = model(**inp).logits[0, -1].float()
            finally:
                if handle is not None: handle.remove()
            return float((lg[R_ID] - lg[L_ID]).item())
        def hs(s, inp):
            with torch.no_grad(): out = model(**inp, output_hidden_states=True)
            return torch.stack([h[0, -1] for h in out.hidden_states]).float().cpu().numpy().astype(np.float16)
        def captured(s, inp, L):                                  # what the hook sees at layer L (for the self-test)
            box = {}
            def hook(mod, args, output): box["h"] = (output[0] if isinstance(output, tuple) else output)[0, -1].float().cpu().numpy()
            hd = LAYERS[L - 1].register_forward_hook(hook)
            try:
                with torch.no_grad(): model(**inp)
            finally: hd.remove()
            return box["h"]
        def readout_axis(s):                                      # direction in the residual stream that changes (logit Right - logit Left): final-norm gain times unembedding difference
            W = model.lm_head.weight.detach(); w = (W[R_ID] - W[L_ID]).float().cpu().numpy()
            gl = [m for n, m in model.named_modules() if n.endswith("language_model.norm") or n == "model.norm"]
            g = gl[0].weight.detach().float().cpu().numpy() if gl else np.ones_like(w); return (g * w).astype(np.float32)
        def positions(s, ia, ib, inone):
            a, b, c = [x["input_ids"][0].cpu().numpy() for x in (ia, ib, inone)]
            assert len(a) == len(b), "donor and recipient prompts differ in token length"
            IMG = getattr(model.config, "image_token_index", None) or getattr(model.config, "image_token_id", None) or proc.tokenizer.convert_tokens_to_ids("<image>")
            img = np.where(a == IMG)[0]; assert len(img) > 100, "image tokens not expanded in input_ids: position bookkeeping would be wrong"
            n, h0 = len(a), int(img.max()) + 1; suf = 0
            while suf < len(c) - 1 and a[n - 1 - suf] == c[len(c) - 1 - suf]: suf += 1
            dig = np.where(a != b)[0].tolist(); region = list(range(h0, n - suf))
            return dict(digits=dig, words=[p for p in region if p not in dig], last=[n - 1], post_hist=list(range(h0, n)))
        def capture(s, inp, Ls, pos):
            with torch.no_grad(): out = model(**inp, output_hidden_states=True)
            assert out.hidden_states[0].shape[1] == inp["input_ids"].shape[1], "hidden-state length != input_ids length"
            idx = torch.tensor(pos)
            return {L: out.hidden_states[L][0].index_select(0, idx.to(out.hidden_states[L].device)).cpu() for L in Ls}
        def delta_pos(s, inp, L, pos, cap, rowmap):
            rows = torch.tensor([rowmap[p] for p in pos]); v0 = cap[L].index_select(0, rows); pidx = torch.tensor(pos)
            def hook(mod, args, output):
                h = output[0] if isinstance(output, tuple) else output
                h = h.clone(); h[:, pidx.to(h.device), :] = v0.to(h.device, h.dtype)
                return (h,) + tuple(output[1:]) if isinstance(output, tuple) else h
            handle = LAYERS[L - 1].register_forward_hook(hook)
            try:
                with torch.no_grad(): lg = model(**inp).logits[0, -1].float()
            finally: handle.remove()
            return float((lg[R_ID] - lg[L_ID]).item())
    RUN = Runner()
print("runner ready")

config.json:   0%|          | 0.00/950 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/70.1k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

... (7 outputs left out)

tokenizer.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/41.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/552 [00:00<?, ?B/s]

language-model layers found at: model.language_model.layers
runner ready


In [6]:
# ===================== SELF-TEST: stop here if anything is off =====================
def inputs(t, with_hist, q=0):
    return RUN.prep(render(T, t, SEED + 1), prompt(history_text(T, t) if with_hist else None, QS[q]))
ok = np.where(FULL & (M.contrast.values >= 0.09))[0]; test_ts = list(ok[:6])
res = []
for t in test_ts:
    dn = (RUN.delta(inputs(t, False, 0)) + RUN.delta(inputs(t, False, 1))) / 2; dh = (RUN.delta(inputs(t, True, 0)) + RUN.delta(inputs(t, True, 1))) / 2
    res.append((dn - M.d_none.values[t], dh - M.d_hist.values[t]))
res = np.array(res); say(f"[self-test 1] reproduction of the cached delta (new - cached): none max|diff| {np.abs(res[:, 0]).max():.3f}, hist max|diff| {np.abs(res[:, 1]).max():.3f}")
assert np.abs(res).max() < 0.35, "cannot reproduce the cached deltas: trials, prompts or model setup differ from the main run"
t = test_ts[0]; inp = inputs(t, True, 0); d0 = RUN.delta(inp)
d_id = RUN.delta(inp, edit=(12, "add", np.zeros(D, np.float32))); say(f"[self-test 2] adding a zero vector changes delta by {abs(d_id - d0):.4f} (should be ~0)")
assert abs(d_id - d0) < 0.05
own = HS[t, 1, 12, :].astype(np.float32); d_own = RUN.delta(inp, edit=(12, "replace", own)); say(f"[self-test 3] replacing the state with the history run's own cached state changes delta by {abs(d_own - d0):.4f} (should be small)")
assert abs(d_own - d0) < 0.25
if not MOCK:
    cap = RUN.captured(inp, 12); cache = HS[t, 1, 12, :].astype(np.float32)
    cs = float(cap @ cache / (np.linalg.norm(cap) * np.linalg.norm(cache))); say(f"[self-test 4] hook state vs cached hidden_states[12]: cosine {cs:.4f} (should be ~1)")
    assert cs > 0.98, "hook indexing does not match hidden_states: layer off by one?"
Lt = 20                                                  # layer 20 lives on the SECOND GPU: this also tests the cross-device 'add' hook
big_vec = (UNIT["evid"][Lt] * 0.5 * RESNORM[Lt]).astype(np.float32); big = RUN.delta(inp, edit=(Lt, "add", big_vec))
say(f"[self-test 5] adding the evidence direction (half the typical residual norm) at layer {Lt} moves delta by {big - d0:+.2f} (should be clearly positive)")
assert big - d0 > (0.01 if MOCK else 0.5), "steering along the evidence direction does nothing: the 'add' hook is broken"
say("self-tests passed")

[self-test 1] reproduction of the cached delta (new - cached): none max|diff| 0.000, hist max|diff| 0.000
[self-test 2] adding a zero vector changes delta by 0.0000 (should be ~0)
[self-test 3] replacing the state with the history run's own cached state changes delta by 0.0000 (should be small)
[self-test 4] hook state vs cached hidden_states[12]: cosine 1.0000 (should be ~1)
[self-test 5] adding the evidence direction (half the typical residual norm) at layer 20 moves delta by +4.30 (should be clearly positive)
self-tests passed


In [7]:
# ===================== SELF-TEST for position patching =====================
def hist_text(k): return f"In the last 10 trials the grating was on the left {10 - k} times and on the right {k} times."
STEER_T = None
rng = np.random.default_rng(2); okc = np.where(M.contrast.values >= 0.0625)[0]
STEER_T = np.sort(np.r_[rng.choice(okc[M.side.values[okc] == "right"], N_STEER // 2, replace=False), rng.choice(okc[M.side.values[okc] == "left"], N_STEER // 2, replace=False)])
def tp_inputs(t):
    img = render(T, t, SEED + 1)
    return RUN.prep(img, prompt(hist_text(2), QS[0])), RUN.prep(img, prompt(hist_text(8), QS[0])), RUN.prep(img, prompt(None, QS[0]))
t = STEER_T[0]; i2, i8, i0 = tp_inputs(t); POS = RUN.positions(i2, i8, i0)
say(f"[self-test 6] positions: digits {POS['digits']} | words {len(POS['words'])} tokens | last {POS['last']} | post_hist {len(POS['post_hist'])} tokens")
if not MOCK:
    ids = i8["input_ids"][0].cpu().numpy(); say("   digit tokens decode to: " + str([proc.tokenizer.decode([int(ids[p])]) for p in POS["digits"]]) + " | first words: " + str([proc.tokenizer.decode([int(ids[p])]) for p in POS["words"][:6]]))
    assert 1 <= len(POS["digits"]) <= 3, "the two prompts should differ at the count digits only"
    assert 8 <= len(POS["words"]) <= 60, "history region has an odd size"
    assert set(POS["digits"]) <= set(POS["post_hist"]) and POS["last"][0] == POS["post_hist"][-1]
d2, d8 = RUN.delta(i2), RUN.delta(i8); say(f"   delta(2 of 10 right) {d2:+.2f} | delta(8 of 10 right) {d8:+.2f}")
rowmap = {p: i for i, p in enumerate(POS["post_hist"])}; cap8 = RUN.capture(i8, [4, 20], POS["post_hist"])
dp = RUN.delta_pos(i2, 4, POS["post_hist"], cap8, rowmap)
say(f"[self-test 7] patching ALL post-image positions at layer 4 of the 2-of-10 run with the 8-of-10 run: delta {dp:+.2f}, should equal the 8-of-10 delta {d8:+.2f}")
assert abs(dp - d8) < (0.01 if MOCK else 0.15), "position patching is broken (hook, positions or layer indexing)"
dz = RUN.delta_pos(i2, 20, POS["last"], RUN.capture(i2, [20], POS["post_hist"]), rowmap) if not MOCK else d2
say(f"[self-test 8] patching a run's own state back changes delta by {abs(dz - d2):.3f} (should be ~0)")
assert abs(dz - d2) < 0.1
say("position self-tests passed")

[self-test 6] positions: digits [599, 606] | words 26 tokens | last [632] | post_hist 52 tokens
   digit tokens decode to: ['2', '8'] | first words: ['', '\n', 'In', 'the', 'last', '']
   delta(2 of 10 right) -0.55 | delta(8 of 10 right) +0.31
[self-test 7] patching ALL post-image positions at layer 4 of the 2-of-10 run with the 8-of-10 run: delta +0.31, should equal the 8-of-10 delta +0.31
[self-test 8] patching a run's own state back changes delta by 0.000 (should be ~0)
position self-tests passed


In [8]:
# ===================== PART 1: directions with controls =====================
READ = RUN.readout_axis(); READ = READ / np.linalg.norm(READ)
def unit(v): return v / (np.linalg.norm(v) + 1e-12)
def perp(v, u): return unit(v - (v @ u) * u)
NAMES2 = ["shift", "shift_perp_evid", "shift_perp_read", "prior_perp_evid", "prior_perp_read", "evid_perp_read"] + [f"shiftshuf{k}" for k in range(1, NSHUF + 1)]
DIR2 = {}; COS = []
for L in STEER2_LAYERS:
    Xn, Xh = HS[:, 0, L, :].astype(np.float32), HS[:, 1, L, :].astype(np.float32); diff = Xh - Xn
    ev, sh, pr = UNIT["evid"][L], UNIT["shift"][L], UNIT["prior"][L]
    d = {"shift": sh, "shift_perp_evid": perp(sh, ev), "shift_perp_read": perp(sh, READ), "prior_perp_evid": perp(pr, ev), "prior_perp_read": perp(pr, READ), "evid_perp_read": perp(ev, READ)}
    for k in range(NSHUF):
        nrp = NR.copy(); nrp[FULL] = NR[FULL][np.random.default_rng(500 + k).permutation(FULL.sum())]       # shuffle the count labels among the full-window trials
        hp, lp = FULL & (nrp >= 8), FULL & (nrp <= 2); d[f"shiftshuf{k + 1}"] = unit((diff[hp].mean(0) - diff[lp].mean(0)) / 2)
    DIR2[L] = {n: (v * NORM[L]).astype(np.float32) for n, v in d.items()}
    allv = dict(d, prior=pr, shuf=UNIT["shuf"][L], evid=ev)
    for n, v in allv.items(): COS.append(dict(layer=L, dir=n, cos_read=float(v @ READ), cos_evid=float(v @ ev), cos_shift=float(v @ sh)))
COS = pd.DataFrame(COS); COS.to_csv(f"{OUT}/cosines.csv", index=False)
say("cosines of each direction with the read-out axis (what changes the answer logit), the evidence axis and the shift direction")
say(COS[~COS.dir.str.startswith("shiftshuf") | COS.dir.isin(["shiftshuf1", "shiftshuf2"])].round(2).to_string(index=False))
sh_ss = COS[COS.dir.str.startswith("shiftshuf")].groupby("layer").agg(read_mean=("cos_read", "mean"), read_abs_max=("cos_read", lambda x: np.abs(x).max()), cos_shift_mean=("cos_shift", "mean"))
say("\nshuffled-shift directions (all %d): cos with read-out axis (mean, max |.|) and with the real shift:" % NSHUF); say(sh_ss.round(2).to_string())

cosines of each direction with the read-out axis (what changes the answer logit), the evidence axis and the shift direction
 layer             dir  cos_read  cos_evid  cos_shift
    16           shift      0.01     -0.17       1.00
    16 shift_perp_evid      0.01     -0.00       0.99
    16 shift_perp_read     -0.00     -0.17       1.00
    16 prior_perp_evid      0.02     -0.00       0.95
    16 prior_perp_read      0.00      0.42       0.79
    16  evid_perp_read      0.00      1.00      -0.17
    16      shiftshuf1      0.00     -0.33      -0.46
    16      shiftshuf2     -0.01     -0.32       0.09
    16           prior      0.03      0.42       0.79
    16            shuf     -0.05     -0.44      -0.43
    16            evid      0.03      1.00      -0.17
    20           shift      0.11      0.21       1.00
    20 shift_perp_evid      0.09      0.00       0.98
    20 shift_perp_read      0.00      0.19       0.99
    20 prior_perp_evid      0.08      0.00       0.94
    20 prior

In [9]:
# ===================== PART 1: run steering (checkpointed) =====================
sp = f"{OUT}/steer2_rows.csv"; sdone = pd.read_csv(sp) if os.path.exists(sp) else pd.DataFrame()
srows = sdone.to_dict("records") if len(sdone) else []; seen = set(zip(sdone.layer, sdone.dir, sdone.s)) if len(sdone) else set()
base_inp = {t: RUN.prep(render(T, t, SEED + 1), prompt(None, QS[0])) for t in STEER_T}
if (0, "base", 0) not in seen:
    for t in STEER_T: srows.append(dict(layer=0, dir="base", s=0, t=t, side=M.side.values[t], contrast=M.contrast.values[t], delta=RUN.delta(base_inp[t])))
    pd.DataFrame(srows).to_csv(sp, index=False)
combos = [(L, d, s) for L in STEER2_LAYERS for d in NAMES2 for s in (-1, 1) if (L, d, s) not in seen]
print(f"steering: {len(STEER_T)} images x {len(combos)} remaining (layer, direction, sign) combinations = {len(STEER_T) * len(combos)} forwards")
t0 = time.time()
for ci, (L, d, s) in enumerate(combos):
    vec = (s * DIR2[L][d]).astype(np.float32)
    for t in STEER_T: srows.append(dict(layer=L, dir=d, s=s, t=t, side=M.side.values[t], contrast=M.contrast.values[t], delta=RUN.delta(base_inp[t], edit=(L, "add", vec))))
    pd.DataFrame(srows).to_csv(sp, index=False); maybe_push("steering2"); el = time.time() - t0
    print(f"  {ci + 1}/{len(combos)} (L{L} {d} {s:+d}x) | {el / 60:.1f} min | ~{el / (ci + 1) * (len(combos) - ci - 1) / 60:.0f} min left", flush=True)
ST = pd.DataFrame(srows); maybe_push("steering2 done", force=True)

steering: 40 images x 72 remaining (layer, direction, sign) combinations = 2880 forwards
[checkpoint] new version ponishbhatia3/llava-pve-b2-ckpt (steering2) | server says: Ok
  1/72 (L16 shift -1x) | 0.4 min | ~29 min left
  2/72 (L16 shift +1x) | 0.8 min | ~27 min left
  3/72 (L16 shift_perp_evid -1x) | 1.2 min | ~27 min left
  4/72 (L16 shift_perp_evid +1x) | 1.6 min | ~26 min left
  5/72 (L16 shift_perp_read -1x) | 1.9 min | ~26 min left
  6/72 (L16 shift_perp_read +1x) | 2.4 min | ~26 min left
  7/72 (L16 prior_perp_evid -1x) | 2.8 min | ~26 min left
  8/72 (L16 prior_perp_evid +1x) | 3.2 min | ~25 min left
  9/72 (L16 prior_perp_read -1x) | 3.6 min | ~25 min left
  10/72 (L16 prior_perp_read +1x) | 4.0 min | ~25 min left
  11/72 (L16 evid_perp_read -1x) | 4.4 min | ~24 min left
  12/72 (L16 evid_perp_read +1x) | 4.8 min | ~24 min left
  13/72 (L16 shiftshuf1 -1x) | 5.2 min | ~23 min left
  14/72 (L16 shiftshuf1 +1x) | 5.6 min | ~23 min left
  15/72 (L16 shiftshuf2 -1x) | 6.0 min 

In [10]:
# ===================== PART 1 analysis =====================
def pick(L, d, s): return ST[(ST.layer == L) & (ST.dir == d) & (ST.s == s)].set_index("t").delta
bh = ST[(ST.dir == "base") & (ST.contrast >= 0.09)]; sw_base = bh[bh.side == "right"].delta.mean() - bh[bh.side == "left"].delta.mean()
def swr(L, d, s):
    x = ST[(ST.layer == L) & (ST.dir == d) & (ST.s == s) & (ST.contrast >= 0.09)]; return (x[x.side == "right"].delta.mean() - x[x.side == "left"].delta.mean()) / sw_base
rg = np.random.default_rng(0); rs = []
for L in STEER2_LAYERS:
    for d in NAMES2:
        diff = ((pick(L, d, 1) - pick(L, d, -1)) / 2).values; bs = [diff[rg.integers(0, len(diff), len(diff))].mean() for _ in range(500)]
        rs.append(dict(layer=L, dir=d, effect=diff.mean(), lo=np.percentile(bs, 2.5), hi=np.percentile(bs, 97.5), frac_of_natural=diff.mean() / NAT_HALF, swing_ratio_plus=swr(L, d, 1), swing_ratio_minus=swr(L, d, -1)))
S2 = pd.DataFrame(rs); S2.to_csv(f"{OUT}/b2b_steer_summary.csv", index=False)
say(f"\n[B2b] steering at 1x natural size, effect = (delta(+dir) - delta(-dir))/2 in logits; natural behavioral half-jump = {NAT_HALF:.2f}")
say(S2[~S2.dir.str.startswith("shiftshuf")].round(2).to_string(index=False))
say("\nshuffled-label shift directions (null): effect per shuffle, and the real shift against them")
for L in STEER2_LAYERS:
    es = S2[(S2.layer == L) & S2.dir.str.startswith("shiftshuf")].effect.values; e0 = S2[(S2.layer == L) & (S2.dir == "shift")].effect.iloc[0]
    sdv = es.std(ddof=1) if len(es) > 1 else np.nan
    say(f"  layer {L}: real shift {e0:+.2f} | shuffled: " + " ".join(f"{x:+.2f}" for x in es) + f" | mean {es.mean():+.2f}, sd {sdv:.2f} | |shuffled| >= |real| in {int((np.abs(es) >= abs(e0)).sum())} of {len(es)} | real minus shuffled mean = {e0 - es.mean():+.2f}")
fig, ax = plt.subplots(1, len(STEER2_LAYERS), figsize=(5 * len(STEER2_LAYERS), 4), squeeze=False)
for j, L in enumerate(STEER2_LAYERS):
    x = S2[S2.layer == L]; ax[0][j].bar(range(len(x)), x.effect, yerr=[x.effect - x.lo, x.hi - x.effect]); ax[0][j].set_xticks(range(len(x))); ax[0][j].set_xticklabels(x.dir, rotation=75, fontsize=7)
    ax[0][j].axhline(NAT_HALF, c="k", ls=":"); ax[0][j].set_title(f"layer {L}: effect on delta (1x)")
plt.tight_layout(); plt.savefig(f"{OUT}/b2b_steer.png", dpi=130); plt.close()


[B2b] steering at 1x natural size, effect = (delta(+dir) - delta(-dir))/2 in logits; natural behavioral half-jump = 0.33
 layer             dir  effect   lo   hi  frac_of_natural  swing_ratio_plus  swing_ratio_minus
    16           shift    0.11 0.10 0.11             0.33              0.99               1.00
    16 shift_perp_evid    0.17 0.16 0.17             0.51              0.99               1.01
    16 shift_perp_read    0.11 0.10 0.11             0.33              0.99               1.00
    16 prior_perp_evid    0.18 0.17 0.19             0.55              0.98               1.01
    16 prior_perp_read    0.30 0.29 0.32             0.93              0.97               1.02
    16  evid_perp_read    0.34 0.33 0.36             1.05              0.97               1.02
    20           shift    0.25 0.24 0.25             0.76              1.01               0.99
    20 shift_perp_evid    0.14 0.13 0.15             0.43              1.02               0.98
    20 shift_perp_read 

In [11]:
# ===================== PART 2: token-position patching (checkpointed) =====================
tp = f"{OUT}/tokpatch_rows.csv"; tdone = pd.read_csv(tp) if os.path.exists(tp) else pd.DataFrame()
trows = tdone.to_dict("records") if len(tdone) else []; have_t = set(tdone.t) if len(tdone) else set()
todo = [t for t in STEER_T if t not in have_t]; t0 = time.time()
print(f"token patching: {len(STEER_T)} images, {len(P_SETS)} position sets x {len(P_LAYERS)} layers x 2 directions per image; images left: {len(todo)}")
for i, t in enumerate(todo):
    i2, i8, i0 = tp_inputs(t); P = RUN.positions(i2, i8, i0); allp = P["post_hist"]; rm = {p: j for j, p in enumerate(allp)}
    d2, d8 = RUN.delta(i2), RUN.delta(i8); cap2, cap8 = RUN.capture(i2, P_LAYERS, allp), RUN.capture(i8, P_LAYERS, allp)
    for L in P_LAYERS:
        for st in P_SETS:
            trows.append(dict(t=t, side=M.side.values[t], contrast=M.contrast.values[t], dir="fwd", set=st, layer=L, d_recip=d2, d_donor=d8, delta=RUN.delta_pos(i2, L, P[st], cap8, rm)))   # recipient 2-of-10, donor 8-of-10
            trows.append(dict(t=t, side=M.side.values[t], contrast=M.contrast.values[t], dir="rev", set=st, layer=L, d_recip=d8, d_donor=d2, delta=RUN.delta_pos(i8, L, P[st], cap2, rm)))
    pd.DataFrame(trows).to_csv(tp, index=False); maybe_push("tokpatch"); el = time.time() - t0
    print(f"  {i + 1}/{len(todo)} | {el / 60:.1f} min | ~{el / (i + 1) * (len(todo) - i - 1) / 60:.0f} min left", flush=True)
TP_ = pd.DataFrame(trows); maybe_push("tokpatch done", force=True)

token patching: 40 images, 4 position sets x 9 layers x 2 directions per image; images left: 40
  1/40 | 0.8 min | ~30 min left
  2/40 | 1.5 min | ~29 min left
  3/40 | 2.3 min | ~28 min left
  4/40 | 3.1 min | ~28 min left
  5/40 | 3.8 min | ~27 min left
  6/40 | 4.6 min | ~26 min left
  7/40 | 5.4 min | ~25 min left
  8/40 | 6.2 min | ~25 min left
  9/40 | 6.9 min | ~24 min left
  10/40 | 7.7 min | ~23 min left
  11/40 | 8.5 min | ~22 min left
  12/40 | 9.2 min | ~22 min left
[checkpoint] new version ponishbhatia3/llava-pve-b2-ckpt (tokpatch) | server says: Ok
  13/40 | 10.1 min | ~21 min left
  14/40 | 10.8 min | ~20 min left
  15/40 | 11.6 min | ~19 min left
  16/40 | 12.4 min | ~19 min left
  17/40 | 13.2 min | ~18 min left
  18/40 | 13.9 min | ~17 min left
  19/40 | 14.7 min | ~16 min left
  20/40 | 15.5 min | ~15 min left
  21/40 | 16.2 min | ~15 min left
  22/40 | 17.0 min | ~14 min left
  23/40 | 17.8 min | ~13 min left
  24/40 | 18.5 min | ~12 min left
  25/40 | 19.3 min | ~1

In [12]:
# ===================== PART 2 analysis =====================
gap = TP_[(TP_.dir == "fwd") & (TP_.set == P_SETS[0]) & (TP_.layer == P_LAYERS[0])]; g = (gap.d_donor - gap.d_recip).values
say(f"\n[B3] delta(8 of 10 right) - delta(2 of 10 right) over {len(g)} images: {g.mean():+.2f} (the 'full' effect that patching should recover)")
rb = np.random.default_rng(0); rows = []
for dr in ("fwd", "rev"):
    for st in P_SETS:
        for L in P_LAYERS:
            x = TP_[(TP_.dir == dr) & (TP_.set == st) & (TP_.layer == L)]; num = (x.delta - x.d_recip).values; den = (x.d_donor - x.d_recip).values
            rec = num.mean() / den.mean(); bs = []
            for _ in range(500): i = rb.integers(0, len(num), len(num)); bs.append(num[i].mean() / den[i].mean())
            rows.append(dict(dir=dr, set=st, layer=L, recovery=rec, lo=np.percentile(bs, 2.5), hi=np.percentile(bs, 97.5)))
R3 = pd.DataFrame(rows); R3.to_csv(f"{OUT}/b3_tokpatch_summary.csv", index=False)
say("recovery = fraction of the 2-vs-8 gap that moves when that set of positions is overwritten at layer L (fwd: 8-of-10 state into a 2-of-10 run; rev: the opposite)")
for dr in ("fwd", "rev"):
    say(f"\n  {dr}:"); piv = R3[R3.dir == dr].pivot(index="layer", columns="set", values="recovery")[P_SETS]; say(piv.round(2).to_string())
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for j, dr in enumerate(("fwd", "rev")):
    for st in P_SETS: x = R3[(R3.dir == dr) & (R3.set == st)]; ax[j].errorbar(x.layer, x.recovery, yerr=[x.recovery - x.lo, x.hi - x.recovery], marker="o", label=st, capsize=2)
    ax[j].axhline(1, c="gray", lw=.5); ax[j].axhline(0, c="gray", lw=.5); ax[j].set_xlabel("layer"); ax[j].set_title(f"B3 {dr}: recovery of the 2-vs-8 gap"); ax[j].legend(fontsize=7)
plt.tight_layout(); plt.savefig(f"{OUT}/b3_tokpatch.png", dpi=130); plt.close()


[B3] delta(8 of 10 right) - delta(2 of 10 right) over 40 images: +0.86 (the 'full' effect that patching should recover)
recovery = fraction of the 2-vs-8 gap that moves when that set of positions is overwritten at layer L (fwd: 8-of-10 state into a 2-of-10 run; rev: the opposite)

  fwd:
set    digits  words  last  post_hist
layer                                
4        0.60   0.12 -0.00        1.0
8        0.18   0.74 -0.00        1.0
10       0.16   0.86 -0.01        1.0
12       0.09   0.65 -0.02        1.0
14       0.05   0.48 -0.03        1.0
16       0.00   0.07  0.30        1.0
18       0.01   0.04  0.73        1.0
20      -0.00   0.02  0.88        1.0
24      -0.00   0.01  0.91        1.0

  rev:
set    digits  words  last  post_hist
layer                                
4        0.89   0.40 -0.00        1.0
8        0.27   0.81 -0.00        1.0
10       0.17   0.86 -0.00        1.0
12       0.04   0.50 -0.02        1.0
14       0.04   0.34 -0.06        1.0
16       0.00   0.

In [13]:
# ===================== SUMMARY: paste everything printed by the previous cells back to Claude =====================
open(f"{OUT}/B2_summary.txt", "w").write("\n".join(SUMMARY)); maybe_push('final', force=True)
with zipfile.ZipFile(f"{OUT}.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for f in os.listdir(OUT): zf.write(f"{OUT}/{f}", f)
print(f"zip written: {OUT}.zip")

[checkpoint] new version ponishbhatia3/llava-pve-b2-ckpt (final) | server says: Ok
zip written: /kaggle/working/B2_out.zip
